In [5]:
prompt = f"""Generate a weekly meal plan based on the following user persona and recipe data:
Only use the recipes exactly as they are presented below. Do not modify the recipes, add any extra information, or create new recipes. 
Simply output the meal name for each day, using only the recipes provided. Do not alter or adapt the meals.
May include those recipes that have disliked ingredients only if the recipes are not sufficient.
If the available recipes are insufficient, you may include recipes that contain disliked ingredients. However, you must strictly follow these rules when checking for disliked ingredients:

1. **ONLY check for disliked ingredients listed in the user's dislikes below. Do NOT flag any ingredient that is NOT in the dislikes list.**
2. **You must NOT infer, assume, or guess the presence of any ingredient. Only check the explicit list of ingredients provided in the recipe.**
3. **If a disliked ingredient (from the list below) is found in a meal, append the following notation to the meal: ' * (contains <disliked ingredient>)'.**
4. **If a meal does not contain any disliked ingredients from the list below, do NOT append anything.**
5. **Do NOT flag Butter, Cheddar Cheese, Yoghurt, Mascarpone Cheese, Mayonnaise, or any other ingredient unless they are explicitly listed in the dislikes section below and the ingredients list of the recipe.**
6. **Ensure that the output is in JSON format only.**

**Meals must be assigned to their respective categories: Breakfast for Breakfast, and Lunch and Dinner should be selected only from the Meal category, with Snacks for Snacks.**

Give response in json format only.
    User Persona:
        Dietary Restrictions: {user_allergens}
        Dislikes: {user_dislikes}
        Likes: {user_likes} 
        Spice Level: Medium
        Age: 29
        Gender: Male
        Height: 180 cm
        Weight: 75 kg 
        Popular Dishes: {user_pref}
        Meal Frequency: 5 meals per day
        Meal Categories: Breakfast, Lunch, Dinner, 2 Snacks  
    """

In [2]:
import os
from dotenv import load_dotenv
from langchain.vectorstores import Pinecone as LangChainPinecone
from langchain.embeddings import HuggingFaceEmbeddings
from pinecone import Pinecone
from langchain_google_genai import ChatGoogleGenerativeAI
import google.generativeai as genai
import json
# from recipe_filter import filter_allergens_in_variants, filter_and_sort_recipes
import ast

# Load environment variables
load_dotenv(override=True)

gemini_api_key = os.getenv('GOOGLE_API_KEY')

# Ensure your Google API key is set
genai.configure(api_key=gemini_api_key)

# Initialize Pinecone
pc = Pinecone(api_key=os.getenv('PINECONE_API_KEY'))
index_name = "recipes-index"

# Load the embedding model (same as used for storing data)
embed_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Connect Pinecone to LangChain
vectorstore = LangChainPinecone(pc.Index(index_name), embed_model, text_key="text")

# Initialize ChatGoogleGenerativeAI for gemini-1.5-flash
llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

# Function to generate responses
def generate_response(prompt):
    model = genai.GenerativeModel("gemini-1.5-flash")
    # Generate content based on the prompt
    response = model.generate_content(prompt)
    return response.text

# Function to filter and sort recipes
def filter_recipes(vectorstore, user_allergens, user_dislikes, query, meal_category, top_k):
    pinecone_filter = {
        "meal_category": {"$eq": meal_category},  # Filter for specific meal type
        "allergens": {"$nin": list(user_allergens)},  # Exclude recipes containing allergens
        "ingredients": {"$nin": list(user_allergens)}  # Exclude recipes containing allergens
    }

    # Retrieve documents from Pinecone with filtering for dislikes and allergens in ingredients
    docs = vectorstore.similarity_search(
        query=query,
        k=top_k,  # Fetch only the required number of results
        filter=pinecone_filter  # Apply the filter for dislikes and allergens in ingredients
    )


    return docs


# Function to format the filtered recipes into a structured meal plan prompt
def format_meal_plan_prompt(filtered_docs, query, user_likes, user_pref):
    prompt = f"""Generate a weekly meal plan based on the following user persona and recipe data:
    Only use the recipes exactly as they are presented below. Do not modify the recipes, add any extra information, or create new recipes. 
    Simply output the meal name for each day, using only the recipes provided. Do not alter or adapt the meals.
    May include those recipes that have disliked ingredients only if the recipes are not sufficient.
    If the available recipes are insufficient, you may include recipes that contain disliked ingredients. However, you must strictly follow these rules when checking for disliked ingredients:

    1. **ONLY check for disliked ingredients listed in the user's dislikes below. Do NOT flag any ingredient that is NOT in the dislikes list.**
    2. **You must NOT infer, assume, or guess the presence of any ingredient. Only check the explicit list of ingredients provided in the recipe.**
    3. **If a disliked ingredient (from the list below) is found in a meal, append the following notation to the meal: ' * (contains <disliked ingredient>)'.**
    4. **If a meal does not contain any disliked ingredients from the list below, do NOT append anything.**
    5. **Do NOT flag Butter, Cheddar Cheese, Yoghurt, Mascarpone Cheese, Mayonnaise, or any other ingredient unless they are explicitly listed in the dislikes section below and the ingredients list of the recipe.**
    6. **Ensure that the output is in JSON format only.**

    **Meals must be assigned to their respective categories: Breakfast for Breakfast, and Lunch and Dinner should be selected only from the Meal category, with Snacks for Snacks.**
    
    Give response in json format only.
        User Persona:
         Dietary Restrictions: {user_allergens}
         Dislikes: {user_dislikes}
         Likes: {user_likes} 
         Spice Level: Medium 
         Popular Dishes: {user_pref}
         Meal Frequency: 5 meals per day
         Meal Categories: Breakfast, Lunch, Dinner, 2 Snacks  
        """
    for i, doc in enumerate(filtered_docs, 1):
        metadata = doc.metadata

        prompt += f"Meal {i}:\n"
        prompt += f" Dish Name: {metadata.get('dish_name', 'Unknown')}\n"
        prompt += f" Description: {metadata.get('description', 'No description')}\n"
        prompt += f" Ingredients: {', '.join(metadata.get('ingredients', []))}\n"
        prompt += f" Spice Level: {metadata.get('spice_level', 'Not specified')}\n"
        prompt += f" Cuisine: {metadata.get('cuisine', 'Unknown')}\n"
        prompt += f" Meal Category: {metadata.get('meal_category', 'Unknown')}\n"
        # prompt += f" Variants: {metadata.get('variants', 'Unknown')}\n"
    
    return prompt

# Main function to generate the meal plan
def generate_meal_plan(vectorstore, user_allergens, user_dislikes, query, user_likes, user_pref):
   
    # Step 1: Fetch exact numbers of recipes for each category directly from Pinecone
    breakfast_docs = filter_recipes(vectorstore, user_allergens, user_dislikes, query, "breakfast", 8)
    meal_docs = filter_recipes(vectorstore, user_allergens, user_dislikes, query, "meal", 16)
    snack_docs = filter_recipes(vectorstore, user_allergens, user_dislikes, query, "snack", 16)

    # Step 2: Organize the filtered recipes by meal category
    # breakfast_docs, meal_docs, snack_docs = organize_recipes_by_category(docs_sorted)
      # Debugging: Print fetched recipe counts
    print(f"Breakfast recipes: {len(breakfast_docs)}")
    print(f"Meal recipes: {len(meal_docs)}")
    print(f"Snack recipes: {len(snack_docs)}")
    print("Fetching complete.")

# Check if the total number of recipes is less than expected
    total_recipes = len(breakfast_docs) + len(meal_docs) + len(snack_docs)
    
    # If the total number of recipes is too few, reset user_allergens and filter again without it
    if total_recipes < (4 + 10 + 10):  # If the number of recipes is less than expected (8 + 16 + 16)
        print("Not enough recipes found. Retrying without allergens filter.")
        user_allergens = {}  # Reset allergens
        # Re-fetch recipes without allergen filter
        breakfast_docs = filter_recipes(vectorstore, user_allergens, user_dislikes, query, "breakfast", 8)
        meal_docs = filter_recipes(vectorstore, user_allergens, user_dislikes, query, "meal", 16)
        snack_docs = filter_recipes(vectorstore, user_allergens, user_dislikes, query, "snack", 16)

    # Step 2: Combine all selected recipes into the final list
    final_docs = breakfast_docs + meal_docs + snack_docs

    # Step 3: Format the structured meal plan prompt
    final_prompt = format_meal_plan_prompt(final_docs, query, user_likes, user_pref)

    # Step 4: Generate the response using LLM
    meal_plan = generate_response(final_prompt)

    print(meal_plan)
    return meal_plan, final_docs

# Example usage
if __name__ == "__main__":
    # User preferences (replace with dynamic input if needed)
    user_allergens = {}  # Example allergens
    # user_dislikes = {
    #     "Cod (white fish)", "Cod Fish", "Cuttlefish", "Fish Sauce",
    #     "Gochujang Paste", "Gochujang Sauce", "Local Wild Fish",
    #     "Nile Perch", "Salmon", "Sea Bass", "Squid", "Tuna",
    #     "White Fish", "Worcestershire Sauce"
    # }  # Example disliked ingredients
    
    # query = "Spice Level: Medium, Cuisine: Mediterranean,European,Comfort Food. Popular Dishes: Classic Chicken Salad, Crudites & Sour Cream Dip, Mini Quiches, Omega Egg Protein Pot"

    # Generate the meal plan
    # generate_meal_plan(vectorstore, user_allergens, user_dislikes, query, user_likes, user_pref)

/home/hello/sb/mealrec/venv/lib/python3.10/site-packages/pinecone/data/index.py:1: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm
/tmp/ipykernel_24953/1996626000.py:25: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embed_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
/tmp/ipykernel_24953/1996626000.py:28: LangChainDeprecationWarning: The class `Pinecone` was deprecated in LangChain 0.0.18 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-pinecone pac

In [9]:
user_allergens = {}  # Example allergens
user_pref = "Classic Chicken Salad, Crudites & Sour Cream Dip, Mini Quiches, Omega Egg Protein Pot"
user_likes = "Mediterranean,European cuisine,Comfort food"
user_dislikes = {
    "Cod (white fish)", "Cod Fish", "Cuttlefish", "Fish Sauce",
    "Gochujang Paste", "Gochujang Sauce", "Local Wild Fish",
    "Nile Perch", "Salmon", "Sea Bass", "Squid", "Tuna",
    "White Fish", "Worcestershire Sauce"
}  # Example disliked ingredients

query = "Spice Level: Medium, Cuisine: Mediterranean,European,Comfort Food. Popular Dishes: Classic Chicken Salad, Crudites & Sour Cream Dip, Mini Quiches, Omega Egg Protein Pot"

# final_docs = generate_meal_plan(vectorstore, user_allergens, user_dislikes, query, user_likes, user_pref)
# final_docs

In [3]:
import pandas as pd

def get_meal_details(dish_name, final_docs):
    """
    Helper function to find meal details from final_docs based on the dish name.
    """
    for doc in final_docs:
        if doc.metadata['dish_name'].strip().lower() == dish_name.strip().lower():
            return doc.metadata
    return {}

import json
import pandas as pd

def generate_meal_data(meal_plan_data, final_docs):
    # Clean up the string: Remove markdown formatting (```json at the start and closing ``` at the end)
    if isinstance(meal_plan_data, str):
        # Remove markdown markers if present
        meal_plan_data = meal_plan_data.strip()  # Strip any surrounding whitespaces
        
        # Check for the markdown formatting
        if meal_plan_data.startswith("```json"):
            meal_plan_data = meal_plan_data[7:].strip()  # Remove the "```json" part
            
        if meal_plan_data.endswith("```"):
            meal_plan_data = meal_plan_data[:-3].strip()  # Remove the ending "```"
        
        # Parse the cleaned-up string into a dictionary
        try:
            meal_plan_data = json.loads(meal_plan_data)
        except json.JSONDecodeError:
            raise ValueError("meal_plan_data is a string but could not be parsed into JSON.")
    
    # Map of days of the week to numbers (for easy lookup)
    day_mapping = {
        "Monday": 1,
        "Tuesday": 2,
        "Wednesday": 3,
        "Thursday": 4,
        "Friday": 5,
        "Saturday": 6,
        "Sunday": 7
    }

    # Initialize a list to store the meal data for DataFrame
    meal_data = []

    # Extract the weekly meal plan from the meal plan data (now a dictionary)
    weekly_meal_plan = meal_plan_data.get("weeklyMealPlan", {})  # Safely get weeklyMealPlan data

    # Function to get meal details from final_docs
    def get_meal_details(dish_name):
        for doc in final_docs:
            if doc.metadata['dish_name'].strip().lower() == dish_name.strip().lower():
                return doc.metadata
        return {}

    # Process each day's meal plan from the weeklyMealPlan data
    for day, meals in weekly_meal_plan.items():
        day_number = day_mapping.get(day, None)  # Get the corresponding day number

        # Loop through each meal type (e.g., Breakfast, Lunch, Snack1, Snack2, Dinner)
        for meal_type, meal_name in meals.items():
            # Clean up meal name: Remove anything after '*' and strip extra spaces
            clean_meal_name = meal_name.split('*')[0].strip()

            # Get meal details from final_docs based on cleaned meal name
            meal_details = get_meal_details(clean_meal_name)
            if meal_details:
                # Add the meal details along with day, day number, and meal type
                meal_details['meal_type'] = meal_type
                meal_details['day'] = day
                meal_details['day_number'] = day_number
                meal_data.append(meal_details)

    # Convert the meal data into a DataFrame
    df = pd.DataFrame(meal_data)

    # Return the resulting DataFrame
    return df



In [4]:
def analyze_meal_data(meal_data_df):
    # Check if the 'cuisine' column exists in the DataFrame
    if 'cuisine' not in meal_data_df.columns:
        raise ValueError("The DataFrame does not contain a 'cuisine' column.")
    
    # Identify the top cuisine preference (most frequent cuisine)
    top_cuisine = meal_data_df['cuisine'].mode()[0]
    print(f"Top Cuisine: {top_cuisine}")
    # Identify the user's most preferred spice level
    top_spice_level = meal_data_df['spice_level'].mode()[0]
    print(f"Top Spice Level: {top_spice_level}")
    
    # Get the count of each cuisine selected by the user
    cuisine_counts = meal_data_df['cuisine'].value_counts().reset_index()
    
    # Rename columns for clarity
    cuisine_counts.columns = ["Cuisine", "Count"]
    
    # Display as a formatted table (this works in Jupyter)
    display(cuisine_counts)




### **User 1**

In [22]:
user_allergens = {}  # Example allergens
user_pref = "Raspberry Yoghurt, Chocolate Muffin, Cheesy Omelette with Broccoli, Cajun Protein with Spinach, Chicken a la King & Onion Bread Roll"
user_likes = "Mediterranean,European cuisine,Comfort food"
user_dislikes = {
    "Chicken",
    "Chicken Stock Powder",
    "Chicken Jus",
    "Chicken Sausage"
}  # Example disliked ingredients

query = "Spice Level: Low, Cuisine: Mediterranean,European,Comfort Food. Popular Dishes: Raspberry Yoghurt, Chocolate Muffin, Cheesy Omelette with Broccoli, Cajun Protein with Spinach, Chicken a la King & Onion Bread Roll"

meal_plan_data, final_docs = generate_meal_plan(vectorstore, user_allergens, user_dislikes, query, user_likes, user_pref)
# final_docs

# Generate the meal data DataFrame using the meal plan and final_docs
meal_data_df = generate_meal_data(meal_plan_data, final_docs)

# Display the resulting DataFrame (top 5 rows)
display(meal_data_df.head())

analyze_meal_data(meal_data_df)


Breakfast recipes: 8
Meal recipes: 16
Snack recipes: 14
Fetching complete.
```json
{
  "weeklyMealPlan": {
    "Monday": {
      "Breakfast": "Med Breakfast Mezze",
      "Snack1": "Panna Cotta with Raspberry",
      "Lunch": "Greek Salad",
      "Snack2": "Omega Egg with Parmesan & Pesto",
      "Dinner": "Beetroot & Quinoa Salad"
    },
    "Tuesday": {
      "Breakfast": "Cilbir",
      "Snack1": "Mini Quiches",
      "Lunch": "Machboos Rubyan",
      "Snack2": "Tomato Basil Soup",
      "Dinner": "Tuscan Protein & Black Rice"
    },
    "Wednesday": {
      "Breakfast": "Classic French Toast",
      "Snack1": "Tuna Nicoise Salad",
      "Lunch": "Peruvian Chicken & Mash Potato",
      "Snack2": "Berries Lemon Cheesecake",
      "Dinner": "Cajun Roasted Chicken Sandwich * (contains Chicken)"
    },
    "Thursday": {
      "Breakfast": "Chicken Patty with Egg * (contains Chicken)",
      "Snack1": "Crudites & Sour Cream Dip",
      "Lunch": "Butter Chickpeas with Rice",
      "Snack2

,allergens,cuisine,description,dish_name,dish_type,ingredients,meal_category,recipe_id,spice_level,variants,meal_type,day,day_number
0,"[Dairy, Gluten, Sesame]",Mediterranean,With pita bread & beetroot hummus. Contains Da...,Med Breakfast Mezze,[Comfort Food],"[Feta Cheese, Cherry Tomato, Garlic, Pita Brea...",breakfast,66950e923d6d34934a274429,Medium,"kcal: 356, carb: 38, fat: 17, protein: 11 | kc...",Breakfast,Monday,1
1,[Dairy],European,-,Panna Cotta with Raspberry,[Chef's Choice],"[Lemon Juice, Sugar, Raspberry, Water, Milk, G...",snack,67a07357c2715c683a3c0848,Low,"kcal: 190, carb: 20, fat: 10, protein: 2",Snack1,Monday,1
2,"[Dairy, Mustard, Soy]",European,Crunchy veggies & feta cheese with olive oil v...,Greek Salad,[Salad],"[Feta Cheese, Cherry Tomato, Black Olives, Gar...",meal,667e02ec013b5ad7ef2909d4,Low,"kcal: 317, carb: 24, fat: 14, protein: 22 | kc...",Lunch,Monday,1
3,"[Celery, Dairy, Eggs, Mustard, Nuts]",Comfort Food,"Boiled eggs with tangy mayo, parmesan & pesto....",Omega Egg with Parmesan & Pesto,[Comfort Food],"[Romaine Lettuce, Egg, Garlic, Basil, Paprika,...",snack,670e8a852794fe87c4084da3,Low,"kcal: 235, carb: 10, fat: 16, protein: 10",Snack2,Monday,1
4,"[Dairy, Nuts, Sesame, Shellfish]",Mediterranean,"With kale, carrot & tahini dressing. Contains ...",Beetroot & Quinoa Salad,[Salad],"[Sunflower Oil, Quinoa, Lemon Juice, Carrot, B...",meal,66796bd9081702ac003a6183,Medium,"kcal: 333, carb: 42, fat: 13, protein: 10 | kc...",Dinner,Monday,1


Top Cuisine: Mediterranean
Top Spice Level: Low


,Cuisine,Count
0,Mediterranean,11
1,European,10
2,Comfort Food,5
3,Asian,3
4,Fusion,2
5,American,2
6,Arabic,1
7,Indian,1


('Mediterranean',
          Cuisine  Count
 0  Mediterranean     11
 1       European     10
 2   Comfort Food      5
 3          Asian      3
 4         Fusion      2
 5       American      2
 6         Arabic      1
 7         Indian      1)

### **User 2**

In [24]:
user_allergens = {}  # Example allergens
user_pref = "Classic Chicken Salad, Grapes Pot"
user_likes = "Mediterranean,European cuisine,Comfort food"
user_dislikes = {
    "Broccoli",
  
}  # Example disliked ingredients

query = "Spice Level: Low, Cuisine: Mediterranean,European,Comfort Food. Popular Dishes: Classic Chicken Salad, Grapes Pot"


meal_plan_data, final_docs = generate_meal_plan(vectorstore, user_allergens, user_dislikes, query, user_likes, user_pref)
# final_docs

# Generate the meal data DataFrame using the meal plan and final_docs
meal_data_df = generate_meal_data(meal_plan_data, final_docs)

# Display the resulting DataFrame (top 5 rows)
display(meal_data_df.head())

analyze_meal_data(meal_data_df)

Breakfast recipes: 8
Meal recipes: 16
Snack recipes: 14
Fetching complete.
```json
{
  "weeklyMealPlan": {
    "Monday": {
      "Breakfast": "Med Breakfast Mezze",
      "Lunch": "Greek Salad",
      "Snack 1": "Classic Chicken Salad",
      "Snack 2": "Kalamata Olives & Parmesan cheese",
      "Dinner": "Beetroot & Quinoa Salad"
    },
    "Tuesday": {
      "Breakfast": "Classic French Toast",
      "Lunch": "Machboos Rubyan",
      "Snack 1": "Tomato Basil Soup",
      "Snack 2": "Berry Smoothie",
      "Dinner": "Tuscan Protein & Black Rice"
    },
    "Wednesday": {
      "Breakfast": "Cilbir",
      "Lunch": "Peruvian Chicken & Mash Potato",
      "Snack 1": "Tuna Nicoise Salad",
      "Snack 2": "Panna Cotta with Raspberry",
      "Dinner": "Lemon Quinoa & Green Beans with Roasted Protein"
    },
    "Thursday": {
      "Breakfast": "Chicken Patty with Egg",
      "Lunch": "Butter Chickpeas with Rice * (contains Broccoli)",
      "Snack 1": "Omega Egg with Parmesan & Pesto",
  

,allergens,cuisine,description,dish_name,dish_type,ingredients,meal_category,recipe_id,spice_level,variants,meal_type,day,day_number
0,"[Dairy, Gluten, Sesame]",Mediterranean,With pita bread & beetroot hummus. Contains Da...,Med Breakfast Mezze,[Comfort Food],"[Feta Cheese, Cherry Tomato, Garlic, Pita Brea...",breakfast,66950e923d6d34934a274429,Medium,"kcal: 356, carb: 38, fat: 17, protein: 11 | kc...",Breakfast,Monday,1
1,"[Dairy, Mustard, Soy]",European,Crunchy veggies & feta cheese with olive oil v...,Greek Salad,[Salad],"[Feta Cheese, Cherry Tomato, Black Olives, Gar...",meal,667e02ec013b5ad7ef2909d4,Low,"kcal: 317, carb: 24, fat: 14, protein: 22 | kc...",Lunch,Monday,1
2,"[Celery, Eggs, Mustard, Nuts]",Mediterranean,"Creamy salad with celery, onions, and lemon. D...",Classic Chicken Salad,[Comfort Food],"[Radicchio Leaves, Romaine Lettuce, Almond]",snack,669763303d6d34934a276688,Medium,"kcal: 136, carb: 8, fat: 5, protein: 12",Snack 1,Monday,1
3,[Dairy],Mediterranean,Olives paired with parmesan cheese. Contains D...,Kalamata Olives & Parmesan cheese,[Chef's Choice],"[Kalamata Olives, Parmesan Cheese]",snack,67581e3387ca72b11cab67d8,Low,"kcal: 146, carb: 1, fat: 12, protein: 6",Snack 2,Monday,1
4,"[Dairy, Nuts, Sesame, Shellfish]",Mediterranean,"With kale, carrot & tahini dressing. Contains ...",Beetroot & Quinoa Salad,[Salad],"[Sunflower Oil, Quinoa, Lemon Juice, Carrot, B...",meal,66796bd9081702ac003a6183,Medium,"kcal: 333, carb: 42, fat: 13, protein: 10 | kc...",Dinner,Monday,1


Top Cuisine: Mediterranean
Top Spice Level: Low


,Cuisine,Count
0,Mediterranean,11
1,European,10
2,Comfort Food,5
3,Asian,3
4,Fusion,2
5,American,2
6,Arabic,1
7,Indian,1


### **User 3**

In [25]:
user_allergens = {}  # Example allergens
user_pref = "Parmesan Soufflé, Salted Almond, Mini Quiches, Tuna Salad, Chicken Crunchy Bowl"
user_likes = "Mediterranean,European cuisine,Comfort food"
user_dislikes = {
    "Papaya","Raw Onion","Onion"
  
}  # Example disliked ingredients

query = "Spice Level: Low, Cuisine: Mediterranean,European,Comfort Food. Popular Dishes: Parmesan Soufflé, Salted Almond, Mini Quiches, Tuna Salad, Chicken Crunchy Bowl"


meal_plan_data, final_docs = generate_meal_plan(vectorstore, user_allergens, user_dislikes, query, user_likes, user_pref)
# final_docs

# Generate the meal data DataFrame using the meal plan and final_docs
meal_data_df = generate_meal_data(meal_plan_data, final_docs)

# Display the resulting DataFrame (top 5 rows)
display(meal_data_df.head())

analyze_meal_data(meal_data_df)

Breakfast recipes: 8
Meal recipes: 16
Snack recipes: 14
Fetching complete.
```json
{
  "weeklyMealPlan": {
    "Monday": {
      "Breakfast": "Med Breakfast Mezze",
      "Lunch": "Greek Salad * (contains Onion)",
      "Snack 1": "Classic Chicken Salad",
      "Dinner": "Tuscan Protein & Black Rice * (contains Onion)",
      "Snack 2": "Mini Quiches"
    },
    "Tuesday": {
      "Breakfast": "Classic French Toast",
      "Lunch": "Beetroot & Quinoa Salad",
      "Snack 1": "Tuna Nicoise Salad",
      "Dinner": "Lemon Quinoa & Green Beans with Roasted Protein * (contains Onion)",
      "Snack 2": "Kalamata Olives & Parmesan cheese"
    },
    "Wednesday": {
      "Breakfast": "Cilbir",
      "Lunch": "Machboos Rubyan",
      "Snack 1": "Omega Egg with Parmesan & Pesto",
      "Dinner": "Peruvian Chicken & Mash Potato",
      "Snack 2": "Tomato Basil Soup"
    },
    "Thursday": {
      "Breakfast": "Greek Yogurt",
      "Lunch": "Butter Chickpeas with Rice * (contains Onion)",
      "

,allergens,cuisine,description,dish_name,dish_type,ingredients,meal_category,recipe_id,spice_level,variants,meal_type,day,day_number
0,"[Dairy, Gluten, Sesame]",Mediterranean,With pita bread & beetroot hummus. Contains Da...,Med Breakfast Mezze,[Comfort Food],"[Feta Cheese, Cherry Tomato, Garlic, Pita Brea...",breakfast,66950e923d6d34934a274429,Medium,"kcal: 356, carb: 38, fat: 17, protein: 11 | kc...",Breakfast,Monday,1
1,"[Dairy, Mustard, Soy]",European,Crunchy veggies & feta cheese with olive oil v...,Greek Salad,[Salad],"[Feta Cheese, Cherry Tomato, Black Olives, Gar...",meal,667e02ec013b5ad7ef2909d4,Low,"kcal: 317, carb: 24, fat: 14, protein: 22 | kc...",Lunch,Monday,1
2,"[Celery, Eggs, Mustard, Nuts]",Mediterranean,"Creamy salad with celery, onions, and lemon. D...",Classic Chicken Salad,[Comfort Food],"[Radicchio Leaves, Romaine Lettuce, Almond]",snack,669763303d6d34934a276688,Medium,"kcal: 136, carb: 8, fat: 5, protein: 12",Snack 1,Monday,1
3,"[Dairy, Fish]",Mediterranean,With garlic parmesan sauce & veggies. Contains...,Tuscan Protein & Black Rice,[Comfort Food],"[Cherry Tomato, Garlic, Onion, Cooking Cream, ...",meal,673837b76b21776aafdb21fb,Low,"kcal: 402, carb: 27, fat: 19, protein: 29 | kc...",Dinner,Monday,1
4,"[Dairy, Eggs]",Comfort Food,"With sundried tomatoes, ricotta, spinach & che...",Mini Quiches,[Comfort Food],"[Egg, Sweet Corn, Spinach, Cheddar Cheese, Sal...",snack,669785453d6d34934a276cb8,Low,"kcal: 182, carb: 8, fat: 11, protein: 11",Snack 2,Monday,1


Top Cuisine: European
Top Spice Level: Low


,Cuisine,Count
0,European,11
1,Mediterranean,10
2,Comfort Food,5
3,Asian,3
4,Fusion,2
5,American,2
6,Indian,1
7,Arabic,1


### **User 4**

In [31]:
user_allergens = {}  # Example allergens
user_pref = "Greek Yogurt, Dates & Mango Bircher Muesli, Yoghurt Granola, Kalamata Olives & Parmesan Cheese"
user_likes = "Mediterranean cuisine,European cuisine,American,Comfort food"
user_dislikes = {
    "Fried Eggs",
    "Mayonnaise"
  
}  # Example disliked ingredients

query = "Spice Level: Low, Cuisine: Mediterranean,European,American,Comfort Food. Popular Dishes: Greek Yogurt, Dates & Mango Bircher Muesli, Yoghurt Granola, Kalamata Olives & Parmesan Cheese"


meal_plan_data, final_docs = generate_meal_plan(vectorstore, user_allergens, user_dislikes, query, user_likes, user_pref)
# final_docs

# Generate the meal data DataFrame using the meal plan and final_docs
meal_data_df = generate_meal_data(meal_plan_data, final_docs)

# Display the resulting DataFrame (top 5 rows)
display(meal_data_df.head())

analyze_meal_data(meal_data_df)

Breakfast recipes: 8
Meal recipes: 16
Snack recipes: 14
Fetching complete.
```json
{
  "weeklyMealPlan": {
    "Monday": {
      "Breakfast": "Med Breakfast Mezze",
      "Snack1": "Kalamata Olives & Parmesan cheese",
      "Lunch": "Greek Salad",
      "Snack2": "Classic Chicken Salad",
      "Dinner": "Machboos Rubyan"
    },
    "Tuesday": {
      "Breakfast": "Greek Yogurt",
      "Snack1": "Kalamata Olives & Parmesan cheese",
      "Lunch": "Tuscan Protein & Black Rice",
      "Snack2": "Tomato Basil Soup",
      "Dinner": "Beetroot & Quinoa Salad"
    },
    "Wednesday": {
      "Breakfast": "Cilbir",
      "Snack1": "Omega Egg with Parmesan & Pesto * (contains Mayonnaise)",
      "Lunch": "Lemon Quinoa & Green Beans with Roasted Protein",
      "Snack2": "Mini Quiches",
      "Dinner": "Tomato Bulgur Pilaf"
    },
    "Thursday": {
      "Breakfast": "Classic French Toast",
      "Snack1": "Kalamata Olives & Parmesan cheese",
      "Lunch": "Asian Rice Noodles With Stir-fry",
  

,allergens,cuisine,description,dish_name,dish_type,ingredients,meal_category,recipe_id,spice_level,variants,meal_type,day,day_number
0,"[Dairy, Gluten, Sesame]",Mediterranean,With pita bread & beetroot hummus. Contains Da...,Med Breakfast Mezze,[Comfort Food],"[Feta Cheese, Cherry Tomato, Garlic, Pita Brea...",breakfast,66950e923d6d34934a274429,Medium,"kcal: 356, carb: 38, fat: 17, protein: 11 | kc...",Breakfast,Monday,1
1,[Dairy],Mediterranean,Olives paired with parmesan cheese. Contains D...,Kalamata Olives & Parmesan cheese,[Chef's Choice],"[Kalamata Olives, Parmesan Cheese]",snack,67581e3387ca72b11cab67d8,Low,"kcal: 146, carb: 1, fat: 12, protein: 6",Snack1,Friday,5
2,"[Dairy, Mustard, Soy]",European,Crunchy veggies & feta cheese with olive oil v...,Greek Salad,[Salad],"[Feta Cheese, Cherry Tomato, Black Olives, Gar...",meal,667e02ec013b5ad7ef2909d4,Low,"kcal: 317, carb: 24, fat: 14, protein: 22 | kc...",Lunch,Monday,1
3,"[Celery, Eggs, Mustard, Nuts]",Mediterranean,"Creamy salad with celery, onions, and lemon. D...",Classic Chicken Salad,[Comfort Food],"[Radicchio Leaves, Romaine Lettuce, Almond]",snack,669763303d6d34934a276688,Medium,"kcal: 136, carb: 8, fat: 5, protein: 12",Snack2,Monday,1
4,"[Dairy, Gluten, Shellfish]",Mediterranean,Traditional Arabic flavored rice with Shrimps....,Machboos Rubyan,[Rice],"[Garlic, Onion, Ghee, Cardamom, Yoghurt, Black...",meal,67a1bf71c2715c683a3cd536,Medium,"kcal: 329, carb: 36, fat: 6, protein: 31 | kca...",Dinner,Monday,1


Top Cuisine: Mediterranean
Top Spice Level: Low


,Cuisine,Count
0,Mediterranean,14
1,European,10
2,Comfort Food,3
3,Asian,2
4,Fusion,2
5,American,2
6,Arabic,1
7,Indian,1


In [33]:
user_allergens = {}  # Example allergens
user_pref = "Grapes Pot, Chocolate Raspberry Mousse, Dates & Mango Bircher Muesli, Omega Egg with Parmesan & Pesto, Tomato Basil Soup"
user_likes = "Mediterranean,European cuisine,American,Comfort food in the given preference"
user_dislikes = {
    "Bell Pepper",
    "Butternut Squash",
    "Peanut Butter",
    "Peanuts",
    "Pumpkin"
  
}  # Example disliked ingredients

query = "Spice Level: Low, Cuisine: Mediterranean,European,American,Comfort Food. Popular Dishes: Grapes Pot, Chocolate Raspberry Mousse, Dates & Mango Bircher Muesli, Omega Egg with Parmesan & Pesto, Tomato Basil Soup"


meal_plan_data, final_docs = generate_meal_plan(vectorstore, user_allergens, user_dislikes, query, user_likes, user_pref)
# final_docs

# Generate the meal data DataFrame using the meal plan and final_docs
meal_data_df = generate_meal_data(meal_plan_data, final_docs)

# Display the resulting DataFrame (top 5 rows)
display(meal_data_df.head())

analyze_meal_data(meal_data_df)

Breakfast recipes: 8
Meal recipes: 16
Snack recipes: 14
Fetching complete.
```json
{
  "weeklyMealPlan": {
    "Monday": {
      "Breakfast": "Med Breakfast Mezze",
      "Snack 1": "Omega Egg with Parmesan & Pesto",
      "Lunch": "Greek Salad",
      "Snack 2": "Tomato Basil Soup",
      "Dinner": "Peruvian Chicken & Mash Potato"
    },
    "Tuesday": {
      "Breakfast": "Classic French Toast",
      "Snack 1": "Kalamata Olives & Parmesan cheese",
      "Lunch": "Beetroot & Quinoa Salad",
      "Snack 2": "Berry Smoothie",
      "Dinner": "Tuscan Protein & Black Rice * (contains Bell Pepper)"
    },
    "Wednesday": {
      "Breakfast": "Waffles with Apple Compote",
      "Snack 1": "Classic Chicken Salad",
      "Lunch": "Lemon Quinoa & Green Beans with Roasted Protein",
      "Snack 2": "Mini Quiches",
      "Dinner": "Machboos Rubyan * (contains Bell Pepper)"
    },
    "Thursday": {
      "Breakfast": "Cilbir",
      "Snack 1": "Tuna Nicoise Salad",
      "Lunch": "Butter Chickp

,allergens,cuisine,description,dish_name,dish_type,ingredients,meal_category,recipe_id,spice_level,variants,meal_type,day,day_number
0,"[Dairy, Gluten, Sesame]",Mediterranean,With pita bread & beetroot hummus. Contains Da...,Med Breakfast Mezze,[Comfort Food],"[Feta Cheese, Cherry Tomato, Garlic, Pita Brea...",breakfast,66950e923d6d34934a274429,Medium,"kcal: 356, carb: 38, fat: 17, protein: 11 | kc...",Breakfast,Monday,1
1,"[Celery, Dairy, Eggs, Mustard, Nuts]",Comfort Food,"Boiled eggs with tangy mayo, parmesan & pesto....",Omega Egg with Parmesan & Pesto,[Comfort Food],"[Romaine Lettuce, Egg, Garlic, Basil, Paprika,...",snack,670e8a852794fe87c4084da3,Low,"kcal: 235, carb: 10, fat: 16, protein: 10",Snack 1,Monday,1
2,"[Dairy, Mustard, Soy]",European,Crunchy veggies & feta cheese with olive oil v...,Greek Salad,[Salad],"[Feta Cheese, Cherry Tomato, Black Olives, Gar...",meal,667e02ec013b5ad7ef2909d4,Low,"kcal: 317, carb: 24, fat: 14, protein: 22 | kc...",Lunch,Monday,1
3,"[Celery, Dairy]",European,Rich tomato soup with basil. Contains Dairy.,Tomato Basil Soup,[Comfort Food],"[Sunflower Oil, Carrot, Basil, Water, Onion, C...",snack,67a1b209c2715c683a3cc6f8,Low,"kcal: 169, carb: 6, fat: 14, protein: 3",Snack 2,Monday,1
4,"[Dairy, Nuts]",Fusion,-,Peruvian Chicken & Mash Potato,[Potato],"[Jalapeno, Garlic, Almond, Red Chilli, Cooking...",meal,679b31687b1396046ad30a42,Medium,"kcal: 689, carb: 65, fat: 24, protein: 52 | kc...",Dinner,Monday,1


Top Cuisine: Mediterranean
Top Spice Level: Low


,Cuisine,Count
0,Mediterranean,11
1,European,10
2,Comfort Food,5
3,Asian,3
4,Fusion,2
5,American,2
6,Indian,1
7,Arabic,1


In [5]:
user_allergens = {"Dairy", "Gluten", "Salt", "Sugar"}  # Example allergens
user_pref = "Greek Yogurt, Dates & Mango Bircher Muesli, Yoghurt Granola, Kalamata Olives & Parmesan Cheese"
user_likes = "Mediterranean cuisine,European cuisine,American,Comfort food"
user_dislikes = {
    "Fried Eggs",
    "Mayonnaise"
  
}  # Example disliked ingredients

query = "Spice Level: Low, Cuisine: Mediterranean,European,American,Comfort Food. Popular Dishes: Greek Yogurt, Dates & Mango Bircher Muesli, Yoghurt Granola, Kalamata Olives & Parmesan Cheese"


meal_plan_data, final_docs = generate_meal_plan(vectorstore, user_allergens, user_dislikes, query, user_likes, user_pref)
# final_docs

# Generate the meal data DataFrame using the meal plan and final_docs
meal_data_df = generate_meal_data(meal_plan_data, final_docs)

# Display the resulting DataFrame (top 5 rows)
display(meal_data_df.head())

analyze_meal_data(meal_data_df)

Breakfast recipes: 1
Meal recipes: 0
Snack recipes: 3
Fetching complete.
Not enough recipes found. Retrying without allergens filter.
```json
{
  "weeklyMealPlan": {
    "Monday": {
      "Breakfast": "Med Breakfast Mezze * (contains Dairy, Gluten, Salt)",
      "Snack 1": "Kalamata Olives & Parmesan cheese * (contains Dairy)",
      "Lunch": "Greek Salad * (contains Dairy, Sugar, Salt)",
      "Snack 2": "Classic Chicken Salad",
      "Dinner": "Machboos Rubyan * (contains Dairy, Salt)"
    },
    "Tuesday": {
      "Breakfast": "Cilbir * (contains Dairy, Gluten, Salt)",
      "Snack 1": "Omega Egg with Parmesan & Pesto * (contains Dairy, Mayonnaise)",
      "Lunch": "Tuscan Protein & Black Rice * (contains Dairy, Salt)",
      "Snack 2": "Tomato Basil Soup * (contains Dairy)",
      "Dinner": "Beetroot & Quinoa Salad * (contains Dairy, Salt)"
    },
    "Wednesday": {
      "Breakfast": "Greek Yogurt * (contains Dairy, Gluten)",
      "Snack 1": "Mini Quiches * (contains Dairy)",
   

,allergens,cuisine,description,dish_name,dish_type,ingredients,meal_category,recipe_id,spice_level,variants,meal_type,day,day_number
0,"[Dairy, Gluten, Sesame]",Mediterranean,With pita bread & beetroot hummus. Contains Da...,Med Breakfast Mezze,[Comfort Food],"[Feta Cheese, Cherry Tomato, Garlic, Pita Brea...",breakfast,66950e923d6d34934a274429,Medium,"kcal: 356, carb: 38, fat: 17, protein: 11 | kc...",Breakfast,Monday,1
1,[Dairy],Mediterranean,Olives paired with parmesan cheese. Contains D...,Kalamata Olives & Parmesan cheese,[Chef's Choice],"[Kalamata Olives, Parmesan Cheese]",snack,67581e3387ca72b11cab67d8,Low,"kcal: 146, carb: 1, fat: 12, protein: 6",Snack 1,Monday,1
2,"[Dairy, Mustard, Soy]",European,Crunchy veggies & feta cheese with olive oil v...,Greek Salad,[Salad],"[Feta Cheese, Cherry Tomato, Black Olives, Gar...",meal,667e02ec013b5ad7ef2909d4,Low,"kcal: 317, carb: 24, fat: 14, protein: 22 | kc...",Lunch,Monday,1
3,"[Celery, Eggs, Mustard, Nuts]",Mediterranean,"Creamy salad with celery, onions, and lemon. D...",Classic Chicken Salad,[Comfort Food],"[Radicchio Leaves, Romaine Lettuce, Almond]",snack,669763303d6d34934a276688,Medium,"kcal: 136, carb: 8, fat: 5, protein: 12",Snack 2,Monday,1
4,"[Dairy, Gluten, Shellfish]",Mediterranean,Traditional Arabic flavored rice with Shrimps....,Machboos Rubyan,[Rice],"[Garlic, Onion, Ghee, Cardamom, Yoghurt, Black...",meal,67a1bf71c2715c683a3cd536,Medium,"kcal: 329, carb: 36, fat: 6, protein: 31 | kca...",Dinner,Monday,1


Top Cuisine: Mediterranean
Top Spice Level: Low


,Cuisine,Count
0,Mediterranean,11
1,European,10
2,Comfort Food,5
3,Asian,3
4,American,2
5,Fusion,2
6,Arabic,1
7,Indian,1
